In [0]:
%python
from pyspark.sql import functions as F

In [0]:
%python
df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "13.223.234.68:9092")
    .option("subscribe", "traffic-data")
    .option("startingOffsets", "earliest")
    .load()
)

In [0]:
%python
bronze = df.select(
    F.col("value").cast("string").alias("message"),
    F.col("topic"),
    F.col("partition"),
    F.col("offset"),
    F.col("timestamp").alias("kafka_timestamp"),
    F.current_timestamp().alias("ingestion_timestamp")
)

In [0]:
%python
query = (
    bronze.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/traffic_catalog/traffic/checkpoints/bronze_traffic"
    )
    .toTable("traffic_catalog.bronze.bronze_traffic")
)

query.awaitTermination()

In [0]:
SELECT *
FROM traffic_catalog.bronze.bronze_traffic
LIMIT 20;

In [0]:
CREATE OR REPLACE TABLE traffic_catalog.bronze.traffic_simulated AS

SELECT
    id,
    CASE (id % 5)
        WHEN 0 THEN 1
        WHEN 1 THEN 2
        WHEN 2 THEN 3
        WHEN 3 THEN 4
        ELSE 5
    END AS road_id,

    CASE (id % 5)
        WHEN 0 THEN 'Rodovia Presidente Dutra'
        WHEN 1 THEN 'Rodovia dos Bandeirantes'
        WHEN 2 THEN 'Rodovia Anhanguera'
        WHEN 3 THEN 'Rodovia Presidente Castello Branco'
        ELSE 'Rodovia Fernão Dias'
    END AS road_name,

    CASE (id % 5)
        WHEN 0 THEN 'BR-116'
        WHEN 1 THEN 'SP-348'
        WHEN 2 THEN 'SP-330'
        WHEN 3 THEN 'SP-280'
        ELSE 'BR-381'
    END AS road_code,

    -23.4 + (rand() * 0.8) AS latitude,
    -46.8 + (rand() * 0.8) AS longitude,

    timestampadd(
        MINUTE,
        id * 10,
        TIMESTAMP '2026-08-01 00:00:00'
    ) AS timestamp,

    CAST(30 + rand() * 80 AS INT) AS current_speed,

    CAST(80 + rand() * 40 AS INT) AS free_flow_speed,

    CAST(300 + rand() * 900 AS INT) AS current_travel_time,

    CAST(300 + rand() * 300 AS INT) AS free_flow_travel_time,

    ROUND(0.70 + rand() * 0.30, 2) AS confidence,

    CASE
        WHEN rand() < 0.02 THEN true
        ELSE false
    END AS road_closure

FROM RANGE(1000);

In [0]:
SELECT *
FROM traffic_catalog.bronze.traffic_simulated
LIMIT 5